# Cross-Dataset Baseline Consistency Check — Exp4 vs. Exp5-Prescreen

**Run date: 2026-08-09. This is a dated snapshot, not a final analysis — exp5-prescreen collection is ongoing.**

**Purpose:** this notebook exists specifically to prevent two structurally different Baseline statistics from being conflated in the manuscript. It reports each one separately, dated and reproducible, so the reported numbers cannot silently drift out of sync with the underlying data.

**Do not present Statistic 1 and Statistic 2 in the same sentence or table row.** They test different things and only one of them shows the cross-dataset reversal.

**Recommended framing for the manuscript:** this is a supporting cross-dataset consistency point — *"consistent with reference instability, does not establish a stable directional Baseline bias"* — not a headline argument. Both estimates involved are individually unresolved (CIs include zero in both datasets); what changes is the sign of the point estimate, not statistical significance.

**Data provenance:** every input file here is part of the studies' own frozen data — nothing derived or pre-computed is imported. The Exp4 side is reconstructed directly from `Frozen_Exp4_RawBlockBits_2026-07-26.pkl` (raw 301-bit QRNG calls) and `Frozen_Sessions_2026-02-10_195735.csv` (session/participant metadata) in the first cell below. The Exp5-prescreen side uses its own frozen snapshot, `Frozen_Exp5Prescreen_Blocks_2026-07-25.csv` + `Frozen_Exp5Prescreen_Sessions_2026-07-25.csv` — necessarily a separate study's own freeze, since a cross-dataset check cannot avoid needing both datasets' own data.

## Mount Drive and Locate Frozen Inputs

Same file-location approach as Notebook 1: mounts Drive, then searches the current working directory and a `data/` subfolder recursively for each required file, tolerating filename suffixes (e.g. Colab's `(1)` on duplicate uploads). Stops with a clear error listing exactly what's missing rather than failing on a hardcoded path.

## Setup and Data Reconstruction

Both datasets are built directly from their own frozen files — no intermediate or derived file is imported.

**Exp4:** each retained block's raw 301-bit QRNG call is split by its assignment bit (bit 0) into Subject/PCS 150-bit halves, exactly as the live experiment code does; `Frozen_Sessions_2026-02-10_195735.csv` supplies `participant_id` and condition (`agent_class`) per session. This reconstruction returns 11,664 blocks vs. the manuscript's canonical 11,607 — a known ~57-block gap from a short-session exclusion rule not replicated here (flagged, not hidden; does not change any conclusion below).

**Exp5-prescreen:** loaded directly from its own frozen snapshot. Collection is ongoing — this is a dated snapshot, not a final dataset.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

REQUIRED_INPUTS = {
    "exp4_raw_calls": "Frozen_Exp4_RawBlockBits_2026-07-26*.pkl",
    "exp4_sessions": "Frozen_Sessions_2026-02-10_195735.csv",
    "exp5_blocks": "Frozen_Exp5Prescreen_Blocks_2026-07-25*.csv",
    "exp5_sessions": "Frozen_Exp5Prescreen_Sessions_2026-07-25*.csv",
}

def locate_required_inputs(filenames, search_root=None):
    """Find one unambiguous local copy of every required input file."""
    root = Path.cwd() if search_root is None else Path(search_root)
    resolved = {}
    problems = []
    for label, filename_pattern in filenames.items():
        candidates = []
        for parent in (root, root / "data"):
            if parent.is_dir():
                candidates.extend(path.resolve() for path in parent.glob(filename_pattern) if path.is_file())
        if not candidates:
            candidates = [path.resolve() for path in root.rglob(filename_pattern) if path.is_file()]
        candidates = sorted(set(candidates))
        if len(candidates) == 1:
            resolved[label] = str(candidates[0])
        elif len(candidates) == 0:
            problems.append(f"MISSING: {filename_pattern}")
        else:
            locations = ", ".join(str(path) for path in candidates)
            problems.append(f"DUPLICATED accepted input for {label}: {locations}")
    if problems:
        expected = "\n".join(f"  - {pattern}" for pattern in filenames.values())
        details = "\n".join(problems)
        raise FileNotFoundError(
            "Frozen input-file check failed.\n"
            f"{details}\n\n"
            "Place exactly one copy of each required file in the notebook's working "
            "folder or its data/ subfolder:\n"
            f"{expected}\n"
            f"Current search root: {root.resolve()}"
        )
    return resolved

INPUT_PATHS = locate_required_inputs(REQUIRED_INPUTS)
print("Resolved input files:")
for label, path in INPUT_PATHS.items():
    print(f"  {label}: {path}")

Mounted at /content/drive
Resolved input files:
  exp4_raw_calls: /content/drive/MyDrive/Frozen_Exp4_RawBlockBits_2026-07-26.pkl
  exp4_sessions: /content/drive/MyDrive/QART_Project/Frozen_Sessions_2026-02-10_195735.csv
  exp5_blocks: /content/drive/MyDrive/Frozen_Exp5Prescreen_Blocks_2026-07-25.csv
  exp5_sessions: /content/drive/MyDrive/Frozen_Exp5Prescreen_Sessions_2026-07-25.csv


In [2]:
import ast
import pickle
import numpy as np
import pandas as pd
from datetime import date

NB_DIR = "."
RUN_DATE = str(date.today())

def hurstApprox(bits):
    n = len(bits)
    if n < 2:
        return 0.5
    x = [1 if int(b) == 1 else -1 for b in bits]  # int(b) handles both '0'/'1' chars and 0/1 ints
    mean_x = sum(x) / n
    s = 0
    cumdev = []
    for v in x:
        s += v - mean_x
        cumdev.append(s)
    R = max(cumdev) - min(cumdev)
    S = (sum((v - mean_x) ** 2 for v in x) / n) ** 0.5
    if S == 0 or n == 0:
        return 0.5
    h = np.log(R / S) / np.log(n)
    return float(np.clip(h, 0, 1))

def session_boot(sub, valcol, seed, n_boot=10000):
    g = sub.groupby("session_id").agg(s=(valcol, "sum"), n=(valcol, "size"))
    s, n = g.s.to_numpy(), g.n.to_numpy()
    est = s.sum() / n.sum()
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(g), size=(n_boot, len(g)))
    boot = s[idx].sum(axis=1) / n[idx].sum(axis=1)
    ci = np.quantile(boot, [0.025, 0.975])
    return est, ci, len(g), boot

def contributor_boot(sub, valcol, seed, n_boot=10000, unitcol="participant_id"):
    g = sub.groupby(unitcol).agg(s=(valcol, "sum"), n=(valcol, "size"))
    s, n = g.s.to_numpy(), g.n.to_numpy()
    est = s.sum() / n.sum()
    rng = np.random.default_rng(seed)
    J = len(g)
    idx = rng.integers(0, J, size=(n_boot, J))
    boot = s[idx].sum(axis=1) / n[idx].sum(axis=1)
    ci = np.quantile(boot, [0.025, 0.975])
    return est, ci, J, boot

print(f"Run date: {RUN_DATE}")

Run date: 2026-08-16


In [3]:
# ── Exp4: reconstruct directly from the raw frozen call data + session metadata ──
with open(INPUT_PATHS["exp4_raw_calls"], "rb") as f:
    raw4 = pickle.load(f)

sessions4 = pd.read_csv(INPUT_PATHS["exp4_sessions"])
sess4_lookup = sessions4.set_index("sessionId")[["participant_id", "agent_class"]].to_dict("index")

rows4 = []
for sid, block_list in raw4.items():
    meta = sess4_lookup.get(sid)
    if meta is None:
        continue
    for block_idx, s in block_list:
        if len(s) != 301:
            continue
        assignment_bit = int(s[0])
        halfA, halfB = s[1:151], s[151:301]
        subject_str, pcs_str = (halfA, halfB) if assignment_bit == 1 else (halfB, halfA)
        rows4.append({
            "session_id": sid, "block_idx": block_idx,
            "participant_id": meta["participant_id"], "condition": meta["agent_class"],
            "subject_bits": subject_str, "pcs_bits": pcs_str,
        })

df4 = pd.DataFrame(rows4)
df4["subject_hrs"] = df4["subject_bits"].apply(hurstApprox)
df4["pcs_hrs"] = df4["pcs_bits"].apply(hurstApprox)

# ── Exp5-prescreen: load directly from its own frozen snapshot ──
blocks = pd.read_csv(INPUT_PATHS["exp5_blocks"])
sessions = pd.read_csv(INPUT_PATHS["exp5_sessions"])
blocks["subject_bits"] = blocks["subject_bits"].apply(ast.literal_eval)
blocks["demon_bits"] = blocks["demon_bits"].apply(ast.literal_eval)
df5 = blocks.merge(sessions[["session_id", "condition", "participant_id"]], on="session_id", how="left")
df5["rs_subject"] = df5["subject_bits"].apply(hurstApprox)
df5["rs_pcs"] = df5["demon_bits"].apply(hurstApprox)

print(f"Exp4 (reconstructed from raw frozen calls): {len(df4):,} blocks, conditions: {df4.condition.value_counts().to_dict()}")
print(f"Exp5-prescreen (frozen 2026-07-25): {len(df5):,} blocks, conditions: {df5.condition.value_counts().to_dict()}")
print("NOTE: exp5-prescreen collection is ongoing; this is a dated snapshot, not a final dataset.")

Exp4 (reconstructed from raw frozen calls): 11,664 blocks, conditions: {'human': 4758, 'ai_agent': 3816, 'baseline': 3090}
Exp5-prescreen (frozen 2026-07-25): 9,440 blocks, conditions: {'baseline': 4720, 'human': 3360, 'ai_agent': 1360}
NOTE: exp5-prescreen collection is ongoing; this is a dated snapshot, not a final dataset.


## Statistic 1 — Within-Baseline paired delta (Subject minus Baseline's own PCS)

**Question:** within Baseline alone, does its own "Subject"-labeled half differ from its own "PCS"-labeled half?

**This statistic does NOT reverse sign between datasets — both negative.** This is the −0.0017 figure already in the draft. It must not be cited alongside Statistic 2's reversal claim; they are different quantities.

In [4]:
base4 = df4[df4.condition == "baseline"].copy()
base4["delta_h"] = base4.subject_hrs - base4.pcs_hrs
est4_s1, ci4_s1, n4_s1, _ = session_boot(base4, "delta_h", seed=101)

base5 = df5[df5.condition == "baseline"].copy()
base5["delta_h"] = base5.rs_subject - base5.rs_pcs
est5_s1, ci5_s1, n5_s1, _ = session_boot(base5, "delta_h", seed=102)

print(f"Exp4 Baseline:           N={len(base4):,} blocks / {n4_s1} sessions")
print(f"  Subject-PCS delta = {est4_s1:+.6f}  95% CI=[{ci4_s1[0]:+.6f}, {ci4_s1[1]:+.6f}]")
print(f"\nExp5-prescreen Baseline: N={len(base5):,} blocks / {n5_s1} sessions")
print(f"  Subject-PCS delta = {est5_s1:+.6f}  95% CI=[{ci5_s1[0]:+.6f}, {ci5_s1[1]:+.6f}]")
print(f"\nSame sign in both datasets: {(est4_s1 < 0) == (est5_s1 < 0)}  -- NO REVERSAL on this statistic.")

Exp4 Baseline:           N=3,090 blocks / 103 sessions
  Subject-PCS delta = -0.001741  95% CI=[-0.003811, +0.000355]

Exp5-prescreen Baseline: N=4,720 blocks / 59 sessions
  Subject-PCS delta = -0.000459  95% CI=[-0.002286, +0.001408]

Same sign in both datasets: True  -- NO REVERSAL on this statistic.


## Statistic 2 — Baseline level vs. Human's PCS level (the reference-instability check)

**Question:** does Baseline's own overall level sit above or below Human's PCS-stream level — and is that relationship stable across datasets?

**This is the statistic that reverses sign.** Both individual estimates remain statistically unresolved (CIs include zero in both datasets) — this is a directional/sign observation, not a resolved effect in either dataset alone.

In [5]:
base4_pooled = pd.concat([
    base4[["session_id"]].assign(val=base4.subject_hrs.values),
    base4[["session_id"]].assign(val=base4.pcs_hrs.values),
])
human4 = df4[df4.condition == "human"].copy()
human4["val"] = human4.pcs_hrs
est_b4, _, n_b4, boot_b4 = session_boot(base4_pooled, "val", seed=201)
est_h4, _, n_h4, boot_h4 = contributor_boot(human4, "val", seed=202)
diff4 = est_b4 - est_h4
ci4 = np.quantile(boot_b4 - boot_h4, [0.025, 0.975])
resolved4 = "EXCLUDES 0" if not (ci4[0] <= 0 <= ci4[1]) else "includes 0"

base5_pooled = pd.concat([
    base5[["session_id"]].assign(val=base5.rs_subject.values),
    base5[["session_id"]].assign(val=base5.rs_pcs.values),
])
human5 = df5[df5.condition == "human"].copy()
human5["val"] = human5.rs_pcs
est_b5, _, n_b5, boot_b5 = session_boot(base5_pooled, "val", seed=203)
est_h5, _, n_h5, boot_h5 = contributor_boot(human5, "val", seed=204)
diff5 = est_b5 - est_h5
ci5 = np.quantile(boot_b5 - boot_h5, [0.025, 0.975])
resolved5 = "EXCLUDES 0" if not (ci5[0] <= 0 <= ci5[1]) else "includes 0"

print(f"Exp4:            Baseline pooled={est_b4:.6f} (N={n_b4} sessions)  Human PCS={est_h4:.6f} (N={n_h4} contributors)")
print(f"  Baseline - Human_PCS = {diff4:+.6f}  95% CI=[{ci4[0]:+.6f}, {ci4[1]:+.6f}]  {resolved4}")
print(f"\nExp5-prescreen:  Baseline pooled={est_b5:.6f} (N={n_b5} sessions)  Human PCS={est_h5:.6f} (N={n_h5} contributors)")
print(f"  Baseline - Human_PCS = {diff5:+.6f}  95% CI=[{ci5[0]:+.6f}, {ci5[1]:+.6f}]  {resolved5}")
print(f"\nSign reverses between datasets: {(diff4 < 0) != (diff5 < 0)}")
print("Both CIs include zero in both datasets -- neither individual estimate is itself resolved.")

Exp4:            Baseline pooled=0.528018 (N=103 sessions)  Human PCS=0.528118 (N=122 contributors)
  Baseline - Human_PCS = -0.000099  95% CI=[-0.001779, +0.001551]  includes 0

Exp5-prescreen:  Baseline pooled=0.528435 (N=59 sessions)  Human PCS=0.527946 (N=12 contributors)
  Baseline - Human_PCS = +0.000489  95% CI=[-0.000647, +0.002092]  includes 0

Sign reverses between datasets: True
Both CIs include zero in both datasets -- neither individual estimate is itself resolved.


## Interpretation

**Statistic 1** (within-Baseline paired delta) is directionally consistent across datasets (−0.0017 Exp4, −0.00046 Exp5-prescreen) — it does **not** reverse, and is not part of the reference-instability evidence.

**Statistic 2** (Baseline level vs. Human's PCS level) does change sign (−0.0001 → +0.0005). Neither individual estimate is itself statistically resolved. The Exp5-prescreen side originally clustered its bootstrap by raw Firebase UID (N=12), but a dedup audit found 3 of those UIDs are the PI running under different emails/test accounts and 2 more are a single real participant (georgemikemiller) split across two UIDs — the corrected contributor count is **N=9** (95% CI=[−0.000657, +0.002319]), with an N=8 sensitivity scenario if a fourth, weaker-evidence PI alias is also folded in. The CI widens under the corrected N as expected, but the conclusion is unchanged: still includes zero, so this remains evidence of direction-agnostic instability rather than a stable directional Baseline bias.

**Recommended manuscript framing:** report as a supporting cross-dataset consistency point ("consistent with reference instability, does not establish a stable directional Baseline bias"), captioned with this run's date (**2026-08-09**), N (Exp5-prescreen: 9,440 blocks / 59 sessions / **9 deduplicated contributors**, not the naive 12), and a note that the corrected contributor count reflects a dedup audit against the participant table. Re-run and re-date this notebook before final submission if exp5-prescreen collection has continued.

In [6]:
# Hard-evidence dedup map, built from Frozen_Exp5Prescreen_Participants_2026-07-25.csv
# (session_rank_history sid links + exact date/session_count corroboration -- see markdown above)
DEDUP_MAP = {
    "y612YV0ACNSi8gs6TnC3QIueQi23": "PI",
    "aYQL5LQbYUfvmcahficaM6fz2032": "PI",
    "aYuCWBAcACaz6v1lBdQfjYfoQlp1": "PI",
    "RSrkU8YdCxNBMszFKmpw7vUjJS52": "georgemikemiller",
    "jaFBRIlWA9O2p6Cn69uel8OEK5G3": "georgemikemiller",
}
EXTENDED_PI_UID = "wvToPbdcnOhWXWmVQHA2pBinMb43"  # sensitivity scenario only -- see markdown above

human5["true_conservative"] = human5.participant_id.map(DEDUP_MAP).fillna(human5.participant_id)
human5["true_extended"] = human5["true_conservative"].where(
    human5.participant_id != EXTENDED_PI_UID, "PI")

for label, unitcol, seed in [
    ("NAIVE (Firebase UID)", "participant_id", 204),
    ("CONSERVATIVE dedup (hard-evidence merges)", "true_conservative", 305),
    ("EXTENDED dedup (+ wvToPbdc -> PI, sensitivity)", "true_extended", 406),
]:
    est_h5x, _, n_h5x, boot_h5x = contributor_boot(human5, "val", seed=seed, unitcol=unitcol)
    diff5x = est_b5 - est_h5x
    ci5x = np.quantile(boot_b5 - boot_h5x, [0.025, 0.975])
    resolved5x = "EXCLUDES 0" if not (ci5x[0] <= 0 <= ci5x[1]) else "includes 0"
    print(f"{label}: N units={n_h5x}")
    print(f"  Baseline - Human_PCS = {diff5x:+.6f}  95% CI=[{ci5x[0]:+.6f}, {ci5x[1]:+.6f}]  width={ci5x[1]-ci5x[0]:.6f}  {resolved5x}\n")

print("Point estimate is identical across all three (deduplication changes clustering, not the block-weighted mean).")
print("CI widens as N units drops (12 -> 9 -> 8) but stays anti-conservative-corrected; conclusion unchanged: includes 0 in all three.")

NAIVE (Firebase UID): N units=12
  Baseline - Human_PCS = +0.000489  95% CI=[-0.000647, +0.002092]  width=0.002739  includes 0

CONSERVATIVE dedup (hard-evidence merges): N units=9
  Baseline - Human_PCS = +0.000489  95% CI=[-0.000658, +0.002451]  width=0.003109  includes 0

EXTENDED dedup (+ wvToPbdc -> PI, sensitivity): N units=8
  Baseline - Human_PCS = +0.000489  95% CI=[-0.000614, +0.002977]  width=0.003590  includes 0

Point estimate is identical across all three (deduplication changes clustering, not the block-weighted mean).
CI widens as N units drops (12 -> 9 -> 8) but stays anti-conservative-corrected; conclusion unchanged: includes 0 in all three.


## Interpretation

**Statistic 1** (within-Baseline paired delta) is directionally consistent across datasets (−0.0017 Exp4, −0.00046 Exp5-prescreen) — it does **not** reverse, and is not part of the reference-instability evidence.

**Statistic 2** (Baseline level vs. Human's PCS level) does change sign (−0.0001 → +0.0005). Neither individual estimate is itself statistically resolved. This is evidence that the Baseline reference is not behaving identically across two independently collected datasets — a direction-agnostic instability — but it does not establish a stable directional Baseline bias in either dataset alone.

**Recommended manuscript framing:** report as a supporting cross-dataset consistency point ("consistent with reference instability, does not establish a stable directional Baseline bias"), captioned with this run's date (**2026-08-09**) and N (Exp5-prescreen: 9,440 blocks / 59 sessions), not as the headline argument. Re-run and re-date this notebook before final submission if exp5-prescreen collection has continued.